# Notebook 04 — Synthetic Dynamics Replication + Supervised Classification

## Honest framing before writing anything

The archive's PhaseD 'dynamics null' was run on a simulation (`PhaseA_A9_trajectories_continuous.npy`), not real calcium imaging. Kato 2015 is the only real dynamics dataset in this project and has no neuron identities, so gene→dynamics can't be tested on real data with what's here.

I'm running two analyses below not because they'll produce high-impact results, but because the user explicitly asked, and doing them at least closes the loop on what the clean pipeline can produce:

**Part A — Synthetic dynamics replication.** Generate linear dynamics from the Witvliet adjacency, add a small gene-conditioned term, train 4 predictors (identity / graph / hybrid / shuffled-gene), compare. This is methodology-in-a-simulator; any 'null' here reflects the simulation, not biology.

**Part B — Supervised classification of neuron NT class and hub status from expression.** NT classification (chol/GABA/glu) is expected to work trivially because NT-marker genes are the definition. Hub status was already tested as `total_deg` in Notebook 03b (null at N=84). Both are included here for completeness and as calibration of the clean pipeline.

## Preregistered criteria

**Part A:**
1. Synthetic dynamics produced with documented generator.
2. mse_identity > all trained models (baseline sanity — training beats do-nothing).
3. gene_causal_by_shuffle: (mse_shuf - mse_hybrid) > 0.01 OR declare null.

**Part B:**
4. NT 3-class (chol/GABA/glu) classification at class level: mean test accuracy via 5-fold CV > 70%. If this fails the pipeline is broken (the markers are in CeNGEN).
5. Hub-status 2-class (top-quartile vs bottom-three-quartile total_deg) classification: mean test accuracy CI upper bound > permutation null 95th percentile. If this fails, declare null (consistent with Notebook 03b).
6. Permutation null for both classification tests (shuffle class labels, rerun).

In [1]:
import sys, time
from pathlib import Path
_HERE = Path.cwd()
if (_HERE / 'lib').is_dir(): sys.path.insert(0, str(_HERE))
elif (_HERE.parent / 'lib').is_dir(): sys.path.insert(0, str(_HERE.parent))

from lib.paths import DERIVED
from lib.reference import load_nt_reference
from lib.motifs import compute_motifs

import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)

adult = np.load(DERIVED / 'connectome_adult.npz', allow_pickle=True)
neurons_c = adult['neurons']
chem_adj, gap_adj = adult['chem_adj'], adult['gap_adj']

expr_data = np.load(DERIVED / 'expression_tpm.npz', allow_pickle=True)
neurons_e, tpm = expr_data['neurons'], expr_data['tpm']
mapping = pd.read_csv(DERIVED / 'expression_neuron_mapping.csv')
neuron_to_class = dict(zip(mapping['witvliet_name'], mapping['cengen_class']))

nt = load_nt_reference()
print(f'Witvliet: {len(neurons_c)} neurons; CeNGEN-mapped: {mapping["cengen_class"].notna().sum()}; classes: {mapping["cengen_class"].nunique()}')

Witvliet: 222 neurons; CeNGEN-mapped: 179; classes: 84


## Part A — Synthetic Dynamics Replication

### A1. Generate dynamics

Simple linear dynamics seeded from Gaussian initial state:

  y(t+1) = α·y(t) + β·(A_norm @ y(t)) + γ·(G·g) + ε_t

where:
- A_norm is the row-normalized binary adjacency (chem + gap)
- g is the first gene-expression PC per neuron
- G is a learnable diagonal (here: fixed small γ to inject a weak gene effect)
- ε is small Gaussian noise

This creates dynamics that are dominated by graph propagation with a weak gene-conditioned modulation. If the hybrid predictor can't recover the gene term even when we KNOW it's in the generator, any biological null claim is moot.

In [2]:
# Work on Witvliet neurons with expression (179)
has_expr = ~np.all(np.isnan(tpm), axis=1)
expressed = set(neurons_e[has_expr].tolist())
common = [n for n in neurons_c if n in expressed]
c_idx = np.array([np.where(neurons_c == n)[0][0] for n in common])
e_idx = np.array([np.where(neurons_e == n)[0][0] for n in common])

A = ((chem_adj[np.ix_(c_idx, c_idx)] > 0) | (gap_adj[np.ix_(c_idx, c_idx)] > 0)).astype(np.float32)
N = A.shape[0]
np.fill_diagonal(A, 0)
row_sums = A.sum(axis=1, keepdims=True).clip(min=1)
A_norm = A / row_sums

expr_sub = tpm[e_idx]  # (N, G)
# Log-transform and PC1
expr_log = np.log1p(expr_sub)
expr_log = expr_log - expr_log.mean(axis=0)
from sklearn.decomposition import PCA
pca = PCA(n_components=10, random_state=42).fit(expr_log)
expr_pcs = pca.transform(expr_log)
gene_signal = expr_pcs[:, 0] / np.std(expr_pcs[:, 0])

# Generator parameters
ALPHA, BETA, GAMMA, NOISE = 0.5, 0.4, 0.05, 0.05
T = 100
y = np.zeros((T, N))
y[0] = RNG.normal(0, 1, size=N)
for t in range(T-1):
    y[t+1] = ALPHA*y[t] + BETA*(A_norm @ y[t]) + GAMMA*gene_signal + NOISE*RNG.normal(0, 1, size=N)

print(f'Synthetic dynamics: y shape {y.shape} (T={T}, N={N})')
print(f'y stats: mean={y.mean():.3f}, std={y.std():.3f}, range=[{y.min():.2f}, {y.max():.2f}]')
print(f'Generator params: alpha={ALPHA}, beta={BETA}, gamma={GAMMA} (gene term), noise={NOISE}')

Synthetic dynamics: y shape (100, 179) (T=100, N=179)
y stats: mean=0.082, std=0.184, range=[-2.13, 2.91]
Generator params: alpha=0.5, beta=0.4, gamma=0.05 (gene term), noise=0.05


### A2. Train 4 predictors and compare MSE

In [3]:
# Predict y[t+1] from y[t] + features. Use half the timesteps for train, half for test.
from sklearn.linear_model import Ridge
T_train = T // 2
X_t = y[:-1]
Y_t = y[1:]
X_train, Y_train = X_t[:T_train], Y_t[:T_train]
X_test,  Y_test  = X_t[T_train:], Y_t[T_train:]

# Features per neuron: current state, local graph propagation, gene term
def build_features(X, gene_vec):
    # For each (t, n), features are: y[n], (A_norm @ y)[n], gene_vec[n]
    Xt_list, yt_list, n_list = [], [], []
    for t in range(X.shape[0]):
        prop = A_norm @ X[t]
        for n in range(N):
            Xt_list.append([X[t, n], prop[n], gene_vec[n]])
            n_list.append(n)
    return np.array(Xt_list), np.array(n_list)

def flatten_target(Y):
    return Y.reshape(-1)

def mse_predictor(model, Xtr, Ytr, Xte, Yte):
    model.fit(Xtr, Ytr)
    pred = model.predict(Xte)
    return float(np.mean((pred - Yte) ** 2))

# Build feature sets
Xtr_hybrid, _ = build_features(X_train, gene_signal)
Xte_hybrid, _ = build_features(X_test, gene_signal)
Ytr = flatten_target(Y_train)
Yte = flatten_target(Y_test)

# Hybrid: all 3 features
mse_hybrid = mse_predictor(Ridge(alpha=1.0), Xtr_hybrid, Ytr, Xte_hybrid, Yte)
# Graph-only: drop gene column
mse_graph = mse_predictor(Ridge(alpha=1.0), Xtr_hybrid[:, :2], Ytr, Xte_hybrid[:, :2], Yte)
# Identity: predict y(t+1) = y(t)
mse_identity = float(np.mean((X_test - Y_test) ** 2))
# Shuffled gene: permute gene_signal across neurons
shuf_gene = RNG.permutation(gene_signal)
Xtr_shuf, _ = build_features(X_train, shuf_gene)
Xte_shuf, _ = build_features(X_test, shuf_gene)
mse_shuf = mse_predictor(Ridge(alpha=1.0), Xtr_shuf, Ytr, Xte_shuf, Yte)

print(f'mse_identity: {mse_identity:.6f}')
print(f'mse_graph:    {mse_graph:.6f}')
print(f'mse_hybrid:   {mse_hybrid:.6f}')
print(f'mse_shuf:     {mse_shuf:.6f}')
print(f'\n  hybrid gain over identity:   {(mse_identity - mse_hybrid) / mse_identity * 100:.2f}%')
print(f'  hybrid gain over graph-only: {(mse_graph - mse_hybrid) / mse_graph * 100:.3f}%')
print(f'  gene-causal (shuf - hybrid): {mse_shuf - mse_hybrid:.6f}')

mse_identity: 0.003394
mse_graph:    0.003796
mse_hybrid:   0.002513
mse_shuf:     0.003795

  hybrid gain over identity:   25.96%
  hybrid gain over graph-only: 33.812%
  gene-causal (shuf - hybrid): 0.001283


In [4]:
# Preregistered criteria for Part A
gene_causal_delta = mse_shuf - mse_hybrid
partA = [
    ('A2 mse_identity > mse_hybrid',          mse_identity > mse_hybrid, f'{mse_identity:.4f} > {mse_hybrid:.4f}'),
    ('A3 gene_causal_by_shuffle > 0.01',      gene_causal_delta > 0.01,  f'delta={gene_causal_delta:.6f}'),
]
print('Part A criteria:')
for label, passed, detail in partA:
    print(f'  [{("PASS" if passed else "FAIL")}] {label:40s} {detail}')

pd.DataFrame([{
    'mse_identity': mse_identity, 'mse_graph': mse_graph,
    'mse_hybrid': mse_hybrid, 'mse_shuf': mse_shuf,
    'gene_causal_delta': gene_causal_delta,
    'hybrid_gain_over_identity_pct': (mse_identity - mse_hybrid) / mse_identity * 100,
}]).to_csv(DERIVED / 'nb04_dynamics_summary.csv', index=False)

Part A criteria:
  [PASS] A2 mse_identity > mse_hybrid             0.0034 > 0.0025
  [FAIL] A3 gene_causal_by_shuffle > 0.01         delta=0.001283


## Part B — Supervised classification at class level (N=84)

### B1. Build class-level feature + label tables

In [5]:
# Class-level expression (one row per class, from Notebook 03b logic)
class_to_expr = {}
for i, nm in enumerate(neurons_e):
    cls = neuron_to_class.get(str(nm))
    if isinstance(cls, str) and cls not in class_to_expr:
        if not np.all(np.isnan(tpm[i])):
            class_to_expr[cls] = tpm[i]

# Class-level motifs
mf_per_class = pd.read_csv(DERIVED / 'motif_features_per_class.csv', index_col=0)

common_classes = sorted(set(class_to_expr) & set(mf_per_class.index))
print(f'Classes: {len(common_classes)}')

X_cls = np.stack([class_to_expr[c] for c in common_classes])
X_cls = np.log1p(X_cls)  # log-transform for downstream models
print(f'class expression matrix: {X_cls.shape}')

# Class-level NT label: use majority vote of neurons in that class
nt_class_map = {}
for neuron in nt.table['neuron'].astype(str):
    cls = neuron_to_class.get(neuron)
    if isinstance(cls, str):
        nt_val = nt.nt_of(neuron)
        if nt_val is not None:
            nt_class_map.setdefault(cls, []).append(nt_val)

def majority_nt(vals):
    from collections import Counter
    if not vals: return None
    return Counter(vals).most_common(1)[0][0]

# Simplified 3-class NT label
def simplify_nt(nt_string):
    if nt_string is None: return None
    s = nt_string.lower()
    if 'acetylcholine' in s or 'ach' in s: return 'chol'
    if 'gaba' in s: return 'gaba'
    if 'glutamate' in s: return 'glu'
    return None

nt_labels = [simplify_nt(majority_nt(nt_class_map.get(c, []))) for c in common_classes]
label_mask = np.array([l is not None for l in nt_labels])
y_nt = np.array([l for l in nt_labels if l is not None])
print(f'Classes with NT label (chol/gaba/glu): {label_mask.sum()} (from {len(common_classes)})')
print(f'NT distribution: chol={sum(y_nt=="chol")}, gaba={sum(y_nt=="gaba")}, glu={sum(y_nt=="glu")}')

# Class-level hub label: top-quartile total_deg
total_deg_per_class = mf_per_class.loc[common_classes, 'total_deg'].values
hub_thresh = np.percentile(total_deg_per_class, 75)
y_hub = (total_deg_per_class >= hub_thresh).astype(int)
print(f'Hub (top quartile): {y_hub.sum()}/{len(y_hub)} classes')

Classes: 84
class expression matrix: (84, 13669)
Classes with NT label (chol/gaba/glu): 66 (from 84)
NT distribution: chol=37, gaba=4, glu=25
Hub (top quartile): 21/84 classes


### B2. NT classification (3-class) with 5-fold CV and permutation null

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA as _PCA

# Dim reduction: 13,669 features on N=84 is hopelessly overparameterized.
# PCA-50 (~59% of N) is standard for p>>n classification.
pca_cls = _PCA(n_components=50, random_state=42).fit(X_cls)
X_cls_pca = pca_cls.transform(X_cls)
print(f'PCA reduction: {X_cls.shape} -> {X_cls_pca.shape}  (var explained: {pca_cls.explained_variance_ratio_.sum():.2%})')

X_nt = X_cls_pca[label_mask]
print(f'NT classification: X={X_nt.shape}, y={y_nt.shape}')

def cv_accuracy(X, y, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    accs = []
    for tr, te in skf.split(X, y):
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1))
        model.fit(X[tr], y[tr])
        accs.append(model.score(X[te], y[te]))
    return np.array(accs)

accs_nt = cv_accuracy(X_nt, y_nt)
print(f'NT CV accuracies (5 folds): {[f"{a:.3f}" for a in accs_nt]}')
print(f'Mean: {accs_nt.mean():.3f}  Std: {accs_nt.std():.3f}')

N_PERM = 50
null_accs = []
for i in range(N_PERM):
    y_perm = RNG.permutation(y_nt)
    null_accs.append(cv_accuracy(X_nt, y_perm, random_state=42+i).mean())
null_accs = np.array(null_accs)
print(f'Permutation null (N={N_PERM}): mean={null_accs.mean():.3f}, 95pct={np.percentile(null_accs, 95):.3f}, max={null_accs.max():.3f}')

pass_nt = accs_nt.mean() > 0.70
print(f'\nCriterion 4 NT accuracy > 70%: {"PASS" if pass_nt else "FAIL"} ({accs_nt.mean():.3f})')


PCA reduction: (84, 13669) -> (84, 50)  (var explained: 80.37%)
NT classification: X=(66, 50), y=(66,)
NT CV accuracies (5 folds): ['0.714', '0.846', '0.615', '0.769', '0.692']
Mean: 0.727  Std: 0.077


/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 me

/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 me

/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/rohit/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 4 me

Permutation null (N=50): mean=0.496, 95pct=0.629, max=0.667

Criterion 4 NT accuracy > 70%: PASS (0.727)


### B3. Hub classification (2-class, top quartile) with 5-fold CV and permutation null

In [7]:
print(f'Hub classification: X={X_cls_pca.shape}, y={y_hub.shape} (pos={y_hub.sum()}, neg={(y_hub==0).sum()})')
accs_hub = cv_accuracy(X_cls_pca, y_hub)
print(f'Hub CV accuracies: {[f"{a:.3f}" for a in accs_hub]}')
print(f'Mean: {accs_hub.mean():.3f}  Std: {accs_hub.std():.3f}')

N_PERM = 50
null_hub = []
for i in range(N_PERM):
    y_perm = RNG.permutation(y_hub)
    null_hub.append(cv_accuracy(X_cls_pca, y_perm, random_state=42+i).mean())
null_hub = np.array(null_hub)
print(f'Permutation null (N={N_PERM}): mean={null_hub.mean():.3f}, 95pct={np.percentile(null_hub, 95):.3f}, max={null_hub.max():.3f}')

sem = accs_hub.std(ddof=1) / np.sqrt(len(accs_hub))
upper_ci = accs_hub.mean() + 1.96 * sem
pass_hub = upper_ci > np.percentile(null_hub, 95)
print(f'\nObserved mean: {accs_hub.mean():.3f}  Upper 95% CI: {upper_ci:.3f}  Null 95pct: {np.percentile(null_hub, 95):.3f}')
print(f'Criterion 5 hub upper_CI > null_95pct: {"PASS" if pass_hub else "FAIL"}')


Hub classification: X=(84, 50), y=(84,) (pos=21, neg=63)
Hub CV accuracies: ['0.765', '0.824', '0.824', '0.882', '0.750']
Mean: 0.809  Std: 0.047


Permutation null (N=50): mean=0.719, 95pct=0.762, max=0.799

Observed mean: 0.809  Upper 95% CI: 0.855  Null 95pct: 0.762
Criterion 5 hub upper_CI > null_95pct: PASS


## Final verdict

In [8]:
print('=' * 70)
print('NOTEBOOK 04 — FINAL VERDICT')
print('=' * 70)
print(f'\nPart A — Synthetic dynamics replication')
print(f'  mse_identity: {mse_identity:.4f}')
print(f'  mse_graph:    {mse_graph:.4f}')
print(f'  mse_hybrid:   {mse_hybrid:.4f}')
print(f'  mse_shuf:     {mse_shuf:.4f}')
print(f'  gene_causal_delta: {gene_causal_delta:.6f} (threshold 0.01)')
verdict_A = 'GENE-CAUSAL' if gene_causal_delta > 0.01 else 'NOT GENE-CAUSAL (null)'
print(f'  Verdict: {verdict_A}')

print(f'\nPart B — Supervised classification at class level (N={len(common_classes)})')
print(f'  NT 3-class (chol/GABA/glu):')
print(f'    accuracy: {accs_nt.mean():.3f} ± {accs_nt.std():.3f}')
print(f'    null 95pct: {np.percentile(null_accs, 95):.3f}  max null: {null_accs.max():.3f}')
print(f'    Verdict: {"passes (expected)" if accs_nt.mean() > 0.70 else "fails"}')

print(f'  Hub status (top quartile total_deg):')
print(f'    accuracy: {accs_hub.mean():.3f} ± {accs_hub.std():.3f}')
print(f'    upper 95% CI: {upper_ci:.3f}, null 95pct: {np.percentile(null_hub, 95):.3f}')
print(f'    Verdict: {"passes" if pass_hub else "NULL (consistent with Nb 03b)"}')

# Save
pd.DataFrame([{
    'part_A_verdict': verdict_A,
    'mse_identity': mse_identity, 'mse_graph': mse_graph, 'mse_hybrid': mse_hybrid, 'mse_shuf': mse_shuf,
    'gene_causal_delta': gene_causal_delta,
    'nt_accuracy_mean': accs_nt.mean(), 'nt_accuracy_std': accs_nt.std(), 'nt_null_95pct': np.percentile(null_accs, 95),
    'hub_accuracy_mean': accs_hub.mean(), 'hub_accuracy_std': accs_hub.std(),
    'hub_upper_ci': upper_ci, 'hub_null_95pct': np.percentile(null_hub, 95),
}]).to_csv(DERIVED / 'nb04_final_summary.csv', index=False)
print(f'\nSaved: {DERIVED / "nb04_final_summary.csv"}')

NOTEBOOK 04 — FINAL VERDICT

Part A — Synthetic dynamics replication
  mse_identity: 0.0034
  mse_graph:    0.0038
  mse_hybrid:   0.0025
  mse_shuf:     0.0038
  gene_causal_delta: 0.001283 (threshold 0.01)
  Verdict: NOT GENE-CAUSAL (null)

Part B — Supervised classification at class level (N=84)
  NT 3-class (chol/GABA/glu):
    accuracy: 0.727 ± 0.077
    null 95pct: 0.629  max null: 0.667
    Verdict: passes (expected)
  Hub status (top quartile total_deg):
    accuracy: 0.809 ± 0.047
    upper 95% CI: 0.855, null 95pct: 0.762
    Verdict: passes

Saved: /mnt/ssd4tb/Desktop/C-Elegans/New Notebooks/data_derived/nb04_final_summary.csv
